# IceNet-MP Pipeline Demo

This demonstration showcases the complete IceNet-MP ML pipeline capabilities through CLI commands.

**Target Audience:** Developer teams and future team members who want to understand our design decisions, 
trade-offs, and flexible experimentation capabilities.

**You'll learn how to:**
- Run our training pipeline end-to-end in three lines of code
- Swap between different modelling paradigms
- Reproduce runs and inspect the outputs

<div class="alert alert-danger">

If you want to run this notebook, you will need a **CDS account** in order to download the ERA5 weather data. Details of how to set this up can be found [here](https://cds.climate.copernicus.eu/how-to-api).

You will also need a **Weights & Biases account** to log the training runs. You can sign up for a free account [here](https://wandb.ai/). Once you have an account, you will need to set up your API key; instructions for doing so can be found [here](https://docs.wandb.ai/models/quickstart#python-notebook).

</div>

## Notebook Structure

[**Section 1: End-to-End Training**](#section-1-end-to-end-training-pipeline)
- Run a full IceNet-MP pipeline end-to-end using a minimal configuration & data
- Inspect training artifacts and see evaluation outputs

[**Section 2: Model Flexibility**](#section-2-model-flexibility)
- Switch between Encode-Process-Decode paradigm and standalone persistence model
- Explore Encoder module functionality (Multimodality)

[**Section 3: Full flexibility - Advanced Example**](#section-3-full-flexibility-advanced-example)
- Write or adapt config files to change pipeline behavior
- Use anemoi functionality to fetch and inspect standard datsets

# Section 1: End-to-End Training Pipeline

In this section, we'll demonstrate the complete training pipeline using a simple **U-Net model with a naive encoder / decoder** (more details of this can be found below in [section 2](#section-2-model-flexibility)). 
The dataset contains sea ice concentration data (OSISAF) and corresponding atmospheric data (ERA5).
We don't expect the model to do well as we are only training for 10 epochs, and won't do any hyperparameter optimisation. However, it will give us a sense of the pipeline.

<div class="alert alert-danger"> 

N.B. to run this example requires you to have a **Weights & Biases account** to log the training runs. You can sign up for a free account [here](https://wandb.ai/). Once you have an account, you will need to set up your API key; instructions for doing so can be found [here](https://docs.wandb.ai/models/quickstart#python-notebook).

</div>


You can install the repo by running the following commands in your terminal:

```bash
git clone https://github.com/alan-turing-institute/icenet_mp
cd icenet_mp
pip install .
```

In [ ]:
# load in the relevant packages / log in to wandb
import os
from IPython.display import Image, Video

!wandb login 

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/ifenton/.netrc.
wandb: Currently logged in as: ifenton (turing-seaice) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Environment Verification

Let's verify that our imp CLI tools are available and working.

To run this notebook, you'll need a kernel (e.g. conda or .venv) with the icenet_mp repo and jupyter installed.

In [2]:
!imp --help

                                                                                
 Usage: imp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 Entrypoint for imp CLI application.                                            
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion            Install completion for the current shell.    │
│ --show-completion               Show completion for the current shell, to    │
│                                 copy it or customize the installation.       │
│ --help                -h        Show this message and exit.                  │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ datasets  Manage datasets 

### Download the dataset for running the model

<div class="alert alert-danger"> 

N.B. to run this minimal example requires you to have signed up to access the ERA5 data. Details on how to do so can be found [here](https://cds.climate.copernicus.eu/how-to-api).

</div>

The model will download a set of ERA5 weather data, and OSISAF sea ice concentration (SIC) data over 2017-2019. The datsets to download are specified in `demo_notebook.yaml`. (Details about how these config files work will be covered in [section 3](#section-3-full-flexibility-advanced-example)).) 

If the data is already present, a summary of the dataset will be printed (it will not be downloaded again). This same summary can be created by running `imp datasets inspect`.

**This downloads the data to a folder called `my_data/` in the root of the repo.**

In [3]:
!imp datasets create --config-name=demo_notebook.yaml

😈 [2026-10-02 12:23:37] Creating dataset samp-floatsouth-argo-25p0km-2020-2024-24h-v3.
😈 [2026-10-02 12:23:37] Dataset <zarr> has missing dates
😈 [2026-10-02 12:23:37] Dataset <zarr> has missing dates
😈 [2026-10-02 12:23:37]   Path    : /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
😈 [2026-10-02 12:23:37]   Progress: 100% (all chunks downloaded).
😈 [2026-10-02 12:23:37] Dataset samp-floatsouth-argo-25p0km-2020-2024-24h-v3 at /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr has been downloaded and seems to be valid.
😈 [2026-10-02 12:23:37] Creating dataset samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1.
😈 [2026-10-02 12:23:37] Dataset <zarr> has missing dates
😈 [2026-10-02 12:23:37] Dataset <zarr> has missing dates
😈 [2026-10-02 12:23:37]   Path    : /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-sicsouth-osisaf-25p

### Train the model

The next command will train a simple U-Net model for sea ice concentration forecasting, and the training will run for 2 epochs. 

In [17]:
!imp train --config-name=demo_notebook.yaml

# save the file path
path_unet_train = os.path.join("../my_data/training/wandb", sorted(os.listdir("../my_data/training/wandb/"))[-1])

😈 [2026-10-02 13:22:00] Anti-aliasing disabled for compatibility with deterministic running and/or accelerator architecture.
😈 [2026-10-02 13:22:00] Building a new 'icenet_mp.models.EncodeProcessDecode' model...
😈 [2026-10-02 13:22:00] Found 3 dataset groups.
😈 [2026-10-02 13:22:00] 1) float-argo:
😈 [2026-10-02 13:22:00]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
😈 [2026-10-02 13:22:00] 2) sic-osisaf:
😈 [2026-10-02 13:22:00]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1.zarr
😈 [2026-10-02 13:22:00] 3) era5:
😈 [2026-10-02 13:22:00]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-weathersouth-era5-25p0km-2020-2024-24h-v4.zarr
😈 [2026-10-02 13:22:00] Dataset /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr has missing dates
😈 [202

### Evaluate the model

Finally the model is evaluated using the checkpoint, saved during the training run. The results are then logged to Weights & Biases. (This will log them to your default WandB project. Within the Sea Ice project there is a WandB account for the Sea Ice project where results are usually logged.)

In [18]:
# get the path to the checkpoint file
ckpt_unet = os.path.join(path_unet_train, "checkpoints/epoch=1-step=722.ckpt")

!imp evaluate --config-name=demo_notebook.yaml --checkpoint="{ckpt_unet}"

# save the file path
path_unet_eval = os.path.join("../my_data/training/wandb", sorted(os.listdir("../my_data/training/wandb/"))[-1])

😈 [2026-10-02 13:30:31] Anti-aliasing disabled for compatibility with deterministic running and/or accelerator architecture.
😈 [2026-10-02 13:30:31] Loading a trained quick-test model...
😈 [2026-10-02 13:30:31] Found 3 dataset groups.
😈 [2026-10-02 13:30:31] 1) float-argo:
😈 [2026-10-02 13:30:31]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
😈 [2026-10-02 13:30:31] 2) sic-osisaf:
😈 [2026-10-02 13:30:31]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1.zarr
😈 [2026-10-02 13:30:31] 3) era5:
😈 [2026-10-02 13:30:31]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-weathersouth-era5-25p0km-2020-2024-24h-v4.zarr
😈 [2026-10-02 13:30:31] Configuring model for evaluation.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try in

We can then check the predictions from this model. (N.b. as it is only run for a small number of epochs on a limited dataset, the results are not great.)

In [19]:
# extract the file name (it has a random string on the end)
img_unet = os.path.join(path_unet_eval, "files/media/videos/output_video")
file = os.path.join(img_unet, os.listdir(img_unet)[0])
Video(file, embed=True)

# Section 2: Model Flexibility

In this section, we'll demonstrate how easy it is to switch between different model architectures.
We'll also show the difference between standalone models and processor models.

The conceptually simpler model type is a standalone model, which takes in the input data and directly outputs the prediction. These models are less flexible, as they have to be specifically coded to handle new input / output data. Consequently a separate instance of the model is likely to be needed for each input / output combination. However, the input variables are available without transformation.  

![pipeline_standalone](../docs/src/assets/pipeline-standalone.png)

The more complex model type is processor model, which uses an encode-process-decode paradigm. Here, the input data is first encoded into a latent representation of fixed dimensions, which is then processed by a core model, before being decoded back into the output space. This allows for more flexibility in terms of input and output variables, as well as the ability to use different types of models for each component.

![pipeline_encode_process_decode](../docs/src/assets/pipeline-encode-process-decode.png)

## Using an alternative processor model

The initial model we ran, the `naive_unet_naive` model, is an example of a processor model. It uses a naive encoder, to convert the input variables into the latent space. A U-Net model is then run on the latent space, before a naive encoder extracts the SIC predictions. 

In this example, we switch the `naive_unet_naive` model for the alternative `cnn_vit_cnn` model, which uses a cnn enocder / decoder, and replaces the U-Net model with a Vision Transformer (ViT).

In [15]:
## Train the model

!imp train --config-name=demo_notebook.yaml model=cnn_vit_cnn 

# save the file path
path_vit_train = os.path.join("../my_data/training/wandb", sorted(os.listdir("../my_data/training/wandb/"))[-1])

😈 [2026-10-02 12:55:48] Anti-aliasing disabled for compatibility with deterministic running and/or accelerator architecture.
😈 [2026-10-02 12:55:48] Building a new 'icenet_mp.models.EncodeProcessDecode' model...
😈 [2026-10-02 12:55:48] Found 3 dataset groups.
😈 [2026-10-02 12:55:48] 1) float-argo:
😈 [2026-10-02 12:55:48]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
😈 [2026-10-02 12:55:48] 2) sic-osisaf:
😈 [2026-10-02 12:55:48]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1.zarr
😈 [2026-10-02 12:55:48] 3) era5:
😈 [2026-10-02 12:55:48]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-weathersouth-era5-25p0km-2020-2024-24h-v4.zarr
😈 [2026-10-02 12:55:48] Dataset /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr has missing dates
😈 [202

In [21]:
## Evaluate the model

# get the path to the checkpoint file
ckpt_vit = os.path.join(path_vit_train, "checkpoints/epoch=1-step=722.ckpt")

!imp evaluate --config-name=demo_notebook.yaml model=cnn_vit_cnn --checkpoint="{ckpt_vit}" 

# save the file path
path_vit_eval = os.path.join("../my_data/training/wandb", sorted(os.listdir("../my_data/training/wandb/"))[-1])

😈 [2026-10-02 14:45:31] Anti-aliasing disabled for compatibility with deterministic running and/or accelerator architecture.
😈 [2026-10-02 14:45:31] Loading a trained cnn-vit-cnn model...
😈 [2026-10-02 14:45:31] Found 3 dataset groups.
😈 [2026-10-02 14:45:31] 1) float-argo:
😈 [2026-10-02 14:45:31]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
😈 [2026-10-02 14:45:31] 2) sic-osisaf:
😈 [2026-10-02 14:45:31]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1.zarr
😈 [2026-10-02 14:45:31] 3) era5:
😈 [2026-10-02 14:45:31]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-weathersouth-era5-25p0km-2020-2024-24h-v4.zarr
😈 [2026-10-02 14:45:32] Configuring model for evaluation.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try i

The results are logged to [Weights & Biases](https://wandb.ai/), but they can also be viewed locally. When inspecting the predictions of the model using a vision transformer architecture you can notice a checkerboard pattern, an artefact of the patch embedding approach this model uses, which occurs when the models are not sufficiently trained.

In [22]:
# extract the file name (it has a random string on the end)
img_vit = os.path.join(path_vit_eval, "files/media/videos/output_video")
file = os.path.join(img_vit, os.listdir(img_vit)[0])
Video(file, embed=True)


There is also a set of other possible model options, however these struggle to train on a laptop, so we won't demonstrate them here. If you want to test them out, see the full list under `config/model` and set the model to your preferred option.

## Standalone persistence model

As mentioned above, an alternative form of model doesn't use an encoder / decoder architecture. We can demonstrate this with a `persistence model` which simply outputs the last input frame as the prediction.

In this case, as well as specifying the model type as `persistence`, we also need to specify a different training config, as the persistence model doesn't need training over multiple epochs.

In [27]:
## Train the model

!imp train --config-name=demo_notebook.yaml train=persistence model=persistence

# save the file path
path_pers_train = os.path.join("../my_data/training/wandb", sorted(os.listdir("../my_data/training/wandb/"))[-1])

😈 [2026-10-02 15:14:39] Anti-aliasing disabled for compatibility with deterministic running and/or accelerator architecture.
😈 [2026-10-02 15:14:39] Building a new 'icenet_mp.models.Persistence' model...
😈 [2026-10-02 15:14:39] Found 3 dataset groups.
😈 [2026-10-02 15:14:39] 1) float-argo:
😈 [2026-10-02 15:14:39]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
😈 [2026-10-02 15:14:39] 2) sic-osisaf:
😈 [2026-10-02 15:14:39]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1.zarr
😈 [2026-10-02 15:14:39] 3) era5:
😈 [2026-10-02 15:14:39]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-weathersouth-era5-25p0km-2020-2024-24h-v4.zarr
😈 [2026-10-02 15:14:39] Dataset /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr has missing dates
😈 [2026-10-02 

In [29]:
## Evaluate the model 
ckpt_pers = os.path.join(path_pers_train, "checkpoints/epoch=2-step=0.ckpt")

!imp evaluate --config-name=demo_notebook.yaml train=persistence model=persistence --checkpoint="{ckpt_pers}"

# save the file path
path_pers_eval = os.path.join("../my_data/training/wandb", sorted(os.listdir("../my_data/training/wandb/"))[-1])

😈 [2026-10-02 15:22:35] Anti-aliasing disabled for compatibility with deterministic running and/or accelerator architecture.
😈 [2026-10-02 15:22:35] Loading a trained persistence model...
😈 [2026-10-02 15:22:35] Found 3 dataset groups.
😈 [2026-10-02 15:22:35] 1) float-argo:
😈 [2026-10-02 15:22:35]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
😈 [2026-10-02 15:22:35] 2) sic-osisaf:
😈 [2026-10-02 15:22:35]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1.zarr
😈 [2026-10-02 15:22:35] 3) era5:
😈 [2026-10-02 15:22:35]    - /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-weathersouth-era5-25p0km-2020-2024-24h-v4.zarr
/Users/ifenton/Documents/Projects/SeaIce/icenet-mp/.venv/lib/python3.11/site-packages/lightning/pytorch/core/saving.py:103: The state dict in PosixPath('/Users/ifenton/Documents/Projects/SeaIce/icenet-

In [30]:
# extract the file name (it has a random string on the end)
img_pers = os.path.join(path_pers_eval, "files/media/videos/output_video")
file = os.path.join(img_pers, os.listdir(img_pers)[0])
Video(file, embed=True)

# Section 3: Full flexibility (Advanced Example)

This section shows how to adapt the different parts of the model pipeline to your needs. We will look in to the config files that are the basis of the pipeline, and we will show how to create your own config file to give you full control over the model training. 

The pipeline uses hydra, which is a powerful configuration management tool. More details can be found [here](https://hydra.cc/docs/intro/). 

These config files are logged as part of the WandB run, so it is easy to go back and check exactly what settings were used for each run, or to reproduce a run.

## The base config file

So far in this notebook, we've mostly used the `demo_notebook.yaml` config file. We can have a look at the contents of this file to see what options it is using.

In [31]:
!cat ../icenet_mp/config/demo_notebook.yaml

defaults:
  - base
  - _self_

train:
  trainer:
    max_epochs: 2

reporting:
  loggers:
    wandb:
      entity: null

base_path: ../my_data


As you can see here, the defaults section points to the `base.yaml` config file, which contains the main options for the pipeline. There are then some specific options for how many epochs to train for, and where to save the data.

So lets have a look at the `base.yaml` file.

In [32]:
!cat ../icenet_mp/config/base.yaml

# Base config inherited by all other configs
defaults:
  - data: sample_south
  - evaluate: default
  - loss: amse
  - model: quick_test
  - predict: sic-osisaf-2d
  - random: default
  - reporting: default
  - train: default
  - platform: default
  - _self_


There are lots more options under the defaults section here, for specifying a range of options such as the datasets, or the model to use. Each of these is another config file, in the folder structure. Some of these options should also look familiar from the `demo_notebook.yaml` file. For example, under `loggers`, we can see `wandb` specified as the logger to use.

We can explore the full list of config folders / files in the `icenet_mp/config/` folder, to get an idea of the options available. 

In [33]:
!ls ../icenet_mp/config/*/

../icenet_mp/config/__pycache__/:
__init__.cpython-311.pyc

../icenet_mp/config/baseline/:
00_climatology.yaml              04_ddpm.yaml
00_persistence.yaml              05_piecewise_unet_piecewise.yaml
01_unet.yaml                     cnn_gsta_cnn.yaml
02_cnn_unet_cnn.yaml             dc_gsta_dc.yaml
03_cnn_vit_cnn.yaml              dc_unet_dc.yaml

../icenet_mp/config/data/:
datasets                 nbsample.yaml            sample_south.yaml
demo.yaml                sample-osisaf.local.yaml split
full_north.yaml          sample_north.local.yaml  synthetic.yaml
full_south.yaml          sample_north.yaml

../icenet_mp/config/evaluate/:
callbacks      default.yaml   synthetic.yaml

../icenet_mp/config/loss/:
amse.yaml         mse.yaml          weighted_bce.yaml
huber.yaml        rmse.yaml         weighted_l1.yaml
mae.yaml          smooth_l1.yaml    weighted_mse.yaml

../icenet_mp/config/model/:
climatology.yaml              dc_unet_dc.yaml
cnn_ddpm_cnn.yaml             ddpm.yaml
cnn_gst

Looking at the contents, you can see that all the defaults in the base config, map to specific folders / files. Each file contains a different set of options that are being used for different parts of the pipeline. You can also see the alternative files that could be substituted in (e.g. using `cnn_vit_cnn.yaml` instead of naive_unet_naive, which is the `quick_test.yaml` for the model).

If you only want to change a single parameter, rather than the whole config file, you can specify this in the config, referencing the nested structure. An example of this is how we specified the number of epochs to train for in the `demo_notebook.yaml` file. Here, we overrode the default of 50 epochs specified in `train/trainer/default.yaml`, by setting `max_epochs` to be 2.

In this file list, you can also see a set of alternative root config files, which we'll look in to in the next section.

### Alternative root config files

Let's look at some of the root config files available in the `configs/` folder. 

In [36]:
!ls ../icenet_mp/config/*.yaml

../icenet_mp/config/base.yaml
../icenet_mp/config/demo_north.yaml
../icenet_mp/config/demo_notebook.yaml
../icenet_mp/config/demo_whale_corridors.yaml
../icenet_mp/config/sample.yaml
../icenet_mp/config/synthetic.yaml


As you can see, the `demo_notebook.yaml` file we've been using is listed here. 

The list of baseline models that we use for testing can be found in the `baseline` folder:

In [42]:
!ls ../icenet_mp/config/baseline/*.yaml

../icenet_mp/config/baseline/00_climatology.yaml
../icenet_mp/config/baseline/00_persistence.yaml
../icenet_mp/config/baseline/01_unet.yaml
../icenet_mp/config/baseline/02_cnn_unet_cnn.yaml
../icenet_mp/config/baseline/03_cnn_vit_cnn.yaml
../icenet_mp/config/baseline/04_ddpm.yaml
../icenet_mp/config/baseline/05_piecewise_unet_piecewise.yaml
../icenet_mp/config/baseline/cnn_gsta_cnn.yaml
../icenet_mp/config/baseline/dc_gsta_dc.yaml
../icenet_mp/config/baseline/dc_unet_dc.yaml


We can see a file called `00_persistence.yaml`.

In [43]:
!cat ../icenet_mp/config/baseline/00_persistence.yaml

# @package _global_
defaults:
  - /base
  - override /train: persistence
  - override /model: persistence
  - _self_


Here you can see that it mostly uses the same parameters as the `base` config. However, it overrides the default configs for `train` and `model` to switch in the persistence model options. 

When we ran the persistence model above, we specified these two `override` settings in the command line. An easier way to run the `persistence` model is to use the `persistence.yaml` config file. However, to run this on your local machine you will need to specify at least the file path to the data folder (the default path is set up for running on a cluster). This could be done with the command:

```bash
imp train --config-name persistence.yaml +data.path=./my_data/
```

<div class="alert alert-success">

Typically you should not change the values in the config files. If you want to change multiple of these options, the best way to do so is to create your own config file, similar to `demo_notebook.yaml`, It's easiest to use the `base` config as the default, and then specify the parameters you want to change using `override` or by specifying the parameter in the config hierarchy. 

When you want to run code using that config file, you can specify it using the `--config-name` option in the CLI command. This will remove the need to specify the individual parameters in the command line each time. 

</div>

## Creating a dataset

The first step in the model pipeline is to download the required input data. This has been developed to make use of the [anemoi datasets](https://anemoi.readthedocs.io/projects/datasets/en/latest/) package, which is part of the [anemoi toolkit](https://anemoi.readthedocs.io/en/latest/) developed by ECMWF. 

New datasets can be downloaded using the `imp datasets create` command shown in the first section of this notebook. 

If the data has already be downloaded, the `imp datasets inspect` command prints a summary of the dataset (or datsets), including a full list of the variables and some summary statistics. 

In [45]:
!imp datasets inspect --config-name=demo_notebook.yaml --verbose

😈 [2026-10-02 15:39:58] Inspecting dataset samp-floatsouth-argo-25p0km-2020-2024-24h-v3.
😈 [2026-10-02 15:39:58] Dataset <zarr> has missing dates
📦 Path          : /Users/ifenton/Documents/Projects/SeaIce/icenet-mp/my_data/data/anemoi/samp-floatsouth-argo-25p0km-2020-2024-24h-v3.zarr
🔢 Format version: 0.19.0

📅 Date      start: 2020-01-01 12:00
📅 Date      end  : 2024-07-31 12:00
⏰ Frequency  : 1d
🚫 Missing    : 819
🌎 Resolution : 25p0km
🌎 Field shape: [432, 432]

📅 Data start : None
📅 Data end   : None

📐 Shape      : 1,674 × 2 × 1 × 186,624 (2.3 GiB)
💽 Size       : 647.9 MiB (647.9 MiB)
📁 Files      : 893


                         Statistics                         
┏━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ Index ┃ Variable ┃     Min ┃  Max ┃ Mean ┃ Stdev ┃ Units ┃
┡━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━┩
│     0 │ PSAL     │    33.7 │ 37.1 │ 34.7 │ 0.486 │ None  │
│     1 │ TEMP     │ 0.00866 │ 28.6 │   12 │  4.94 │ None  │
└───────┴────────

The details of the dataset to be downloaded are defined in the datasets config file. The default datasets (those used by the `base` config) are `samp_sicsouth_osisaf_25p0km_2020_2024_24h_v1`, `samp_weathersouth_era5_25p0km_2020_2024_24h_v4` and `samp_floatsouth_argo_25p0km_2020_2024_24h_v3`. We will have a look at these in a bit more detail.

In [48]:
!cat ../icenet_mp/config/data/datasets/samp_sicsouth_osisaf_25p0km_2020_2024_24h_v1.yaml

samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1:
  name: samp-sicsouth-osisaf-25p0km-2020-2024-24h-v1
  group_as: sic-osisaf
  description: Sea ice concentration from SMMR/SSMI/SSMIS + AMSR-E/AMSR2 + AMSR2 for southern hemisphere
  attribution: EUMETSAT/OSISAF
  licence: CC-BY-4.0

  dates:
    start: '2020-01-01T12:00:00'
    end: '2024-07-31T12:00:00'
    frequency: 24h
    # To reduce size, we only include the data ranges needed by the sample split
    missing:
    - start: 2022-01-01T12:00:00
      end: 2022-06-30T12:00:00
    - start: 2022-08-01T12:00:00
      end: 2022-12-31T12:00:00
    - start: 2023-02-01T12:00:00
      end: 2023-12-31T12:00:00
    - start: 2024-02-01T12:00:00
      end: 2024-06-30T12:00:00

  postprocessors:
    mask_generator:
      _target_: icenet_mp.ingestion.postprocessors.StatusFlagMaskGenerator

  # Keep normalisation statistics within the training split.
  statistics:
    end: '2021-12-31T12:00:00'

  input:
    pipe:
      - concat:
        # OSI-458 [AM

This dataset contains sea ice concentration (SIC) data from the OSI SAF 25km dataset for the years 2020-2024, with a temporal resolution of 24 hours. The data is from the Southern Hemisphere. This download makes use of data preprocessing developed as part of the [IceNet repository](https://github.com/icenet-ai/icenet).

In [49]:
!cat ../icenet_mp/config/data/datasets/samp_weathersouth_era5_25p0km_2020_2024_24h_v4.yaml

samp-weathersouth-era5-25p0km-2020-2024-24h-v4:
  name: samp-weathersouth-era5-25p0km-2020-2024-24h-v4
  group_as: era5
  description: Weather from ERA5 for partial southern hemisphere
  attribution: ECMWF/C3S
  licence: CC-BY-4.0

  dates:
    start: '2020-01-01T12:00:00'
    end: '2024-07-31T12:00:00'
    frequency: 24h
    # To reduce size, we only include the data ranges needed by the sample split
    missing:
    - start: 2022-01-01T12:00:00
      end: 2022-06-30T12:00:00
    - start: 2022-08-01T12:00:00
      end: 2022-12-31T12:00:00
    - start: 2023-02-01T12:00:00
      end: 2023-12-31T12:00:00
    - start: 2024-02-01T12:00:00
      end: 2024-06-30T12:00:00

  # Keep normalisation statistics within the training split.
  statistics:
    end: '2021-12-31T12:00:00'

  input:
    pipe:
      - join:
        - mars:
            use_cdsapi_dataset: "reanalysis-era5-complete"
            area: "-15/-180/-90/180"  # southern hemisphere
            grid: "0.25 / 0.25"
            levtyp

This dataset contains 50km resolution weather data from ERA5 for the years 2020-2024, with a temporal resolution of 24 hours. The data is from the Southern Hemisphere. The variables downloaded are those commonly used in sea ice forecasting, including temperature, u and v wind components, surface / sea level pressure, humidity and geopotential height. It also includes the cos and sin julian day to help the model learn seasonal patterns.

N.b. there are also a set of other configs in the `config/datasets` folder for different subsets of variables, different hemispheres, and different datasets. 

## Training a model

Training a model is done using the `imp train` command, as shown in [section 1](#train-the-model) of this notebook. There are a set of config files that specifically relate to the model training. For example the choice of model, the training parameters, the variable to predict, the dataset split and the logger to use. We will explore each of these in turn.

### Model choice

In [50]:
!cat ../icenet_mp/config/model/quick_test.yaml

_target_: icenet_mp.models.EncodeProcessDecode

name: quick-test

encoders:
  latent_space: [128, 128]
  era5:
    _target_: icenet_mp.models.encoders.NaiveLinearEncoder
  float-argo:
    _target_: icenet_mp.models.encoders.NaiveLinearEncoder
  sic-osisaf:
    _target_: icenet_mp.models.encoders.NaiveLinearEncoder
  sic-ssmis:
    _target_: icenet_mp.models.encoders.NaiveLinearEncoder

processor:
  _target_: icenet_mp.models.processors.UNetProcessor
  start_out_channels: 64

decoder:
  _target_: icenet_mp.models.decoders.NaiveLinearDecoder
  mask_type: active
  restrict_range: sigmoid
  skip_connection:
    method: none


This config points to the specific python code used for each part of the model (i.e. the naive encoder, the U-Net processor and the naive decoder). There are various parameters that can be set, such as the size of the latent space and the number of start-out channels for the U-Net. 

There are a set of possible model configs - each of these will have specific parameters that need to be set for those models, and their default values. 

### Training parameters

In [51]:
!cat ../icenet_mp/config/train/default.yaml

defaults:
  - callbacks:
    - best_checkpoint
    - early_stopping
    - ema_weight_averaging
    - learning_rate
    - media_logging
    - metric_summary
  - lr_scheduler: default
  - multistage: default
  - optimizer: default
  - scheduler: default
  - trainer: default
  - _self_


A set of training parameters are specified here, normally it is fine to just use the default values. 

### Prediction window parameters

In [ ]:
!cat ../icenet_mp/config/window/forecast-3-history-2.yaml

The `variables` config key specifies the variable(s) to be predicted, while the `window` config key specifies the number of historic days to include as input, and the number of days to forecast ahead. There are config files for predicting SIC in the northern or the southern hemisphere.

### Dataset split

In [57]:
!cat ../icenet_mp/config/data/split/sample_dataset.yaml

batch_size: 2

# Holdout set is not applicable for sample data
predict:
  - start: null
    end: null

# This data is an evaluation set used to compare fully-trained models (e.g. for releases)
# Data from 2024 (2 months) [strict subset of full]
test:
  - start: 2024-01-01
    end: 2024-01-31
  - start: 2024-07-01
    end: 2024-07-31

# This data is used for training the model
# Data from 2020-2021 (24 months) [strict subset of full]
train:
  - start: 2020-01-01
    end: 2021-12-31

# This data is used for validation during training (e.g. for early stopping etc.)
# Data from 2022 and 2023 (2 months) [strict subset of full]
validate:
  - start: 2022-07-01
    end: 2022-07-31
  - start: 2023-01-01
    end: 2023-01-31


The dataset split specifies the number of batches to use for training. It also specifies how the datasets should be split in to training, validation and test sets, based on date ranges. There are different splits for the sample or the full dataset, however the dates shouldn't be altered, to ensure that all the different model runs are comparable.

### Loggers

In [59]:
!cat ../icenet_mp/config/reporting/loggers/wandb.yaml

wandb:
  _target_: lightning.pytorch.loggers.wandb.WandbLogger
  entity: "turing-seaice"
  log_model: false # set 'true' to log the model to W&B (requires sufficient storage space)
  name: null # Run name: defaults to random, can be overridden by the user
  offline: false # set 'true' to run locally instead of uploading to W&B
  save_dir: ${base_path}/training/


By default, the pipeline uses Weights & Biases to log the training and evaluation metrics. All runs are saved to the 'turing-seaice' project, as well as being saved locally in a `training` folder alongside the data. We don't log the model weights to WandB to save space.

## Evaluating a model

Training a model produces a set of artifacts, including one or more model checkpoint. The checkpoints can be used to evaluate the model performance using the `imp evaluate` command (as shown in [section 1](#evaluate-the-model) of this notebook). There are a couple of config files that specifically relate to the evaluation process within the `evaluate` folder (though several of the ones we have already explored are also relevant).

### Plotting

In [61]:
!cat ../icenet_mp/config/evaluate/callbacks/media_logging.yaml

media_logging:
  _target_: icenet_mp.callbacks.MediaLoggingCallback
  frequency:
    number: 3 # plot exactly 3 batches equally spaced across the epoch
  make_input_plots: false
  make_static_plots: true
  make_video_plots: true
  model_name: ${model.name}
  plot_spec:
    include_ice_edge: true


By default, static maps and videos of the predicted forecasts of the output variable (normally SIC) are created for a set of examples from the test set. These are uploaded to Weights & Biases, and also saved locally. 

There are a lot of options for adjusting the model output plots. 

### Metric summary

In [62]:
!cat ../icenet_mp/config/evaluate/callbacks/metric_summary.yaml

metric_summary:
  _target_: icenet_mp.callbacks.MetricSummaryCallback


The default metric summary is based on average loss.

<div class="alert alert-success">

Typically you should not change the values in the config files. If you want to change multiple of these options, the best way to do so is to create your own config file, similar to `demo_notebook.yaml`, It's easiest to use the `base` config as the default, and then specify the parameters you want to change using `override` or by specifying the parameter in the config hierarchy. 

When you want to run code using that config file, you can specify it using the `--config-name` option in the CLI command. This will remove the need to specify the individual parameters in the command line each time. 

</div>

Hopefully this notebook has given you an overview of the Sea Ice forecasting pipeline, and shown you how you can use it flexibily to run different models and evaluate their performance.